In [2]:
import random
import os
import time
from IPython.display import clear_output
import heapq

In [3]:
def printG(grid):
    for i in range(0, 9, 3):
        print(grid[i:i+3])
    print()

In [4]:
def clear_screen():
    os.system('clear')

In [5]:
class Puzzle():
    def __init__(self, grid, goal):
        """Initializes the puzzle with a starting grid and a goal state."""
        self.grid = grid[:]
        self.goal = goal

    def find_index(self, num):
        """Finds the index of a given number in the grid."""
        for i, val in enumerate(self.grid):
            if val == num:
                return i
        return -1

    def get_valid_moves(self):
        """Returns a list of valid moves"""
        i = self.find_index(0)
        valid_moves = []
        
        if i >= 3:
            valid_moves.append("up")
        if i <= 5:
            valid_moves.append("down")
        if i % 3 != 0:
            valid_moves.append("left")
        if i % 3 != 2:
            valid_moves.append("right")
            
        return valid_moves

    def move(self, direction):
        """Moves the empty tile in the specified direction."""
        i = self.find_index(0)
        
        if direction not in self.get_valid_moves():
            return

        if direction == "up":
            swap_index = i - 3
        elif direction == "down":
            swap_index = i + 3
        elif direction == "left":
            swap_index = i - 1
        elif direction == "right":
            swap_index = i + 1
        
        self.grid[i], self.grid[swap_index] = self.grid[swap_index], self.grid[i]

In [6]:
# ==================== HEURISTIC FUNCTION ====================
def calculate_manhattan_distance(grid, goal):
    """Calculate Manhattan distance heuristic."""
    distance = 0
    for i in range(9):
        if grid[i] != 0:
            row_current, col_current = divmod(i, 3)
            row_goal, col_goal = divmod(goal.index(grid[i]), 3)
            distance += abs(row_current - row_goal) + abs(col_current - col_goal)
    return distance

In [ ]:
Grid = [7, 2, 4, 5, 0, 6, 8, 3, 1]
Goal = [0, 1, 2, 3, 4, 5, 6, 7, 8]

def ReflexAgent():
    """A simple reflex agent - random moves."""
    puzzle = Puzzle(Grid, Goal)
    moves_count = 0
    
    print("Initial Grid:")
    printG(puzzle.grid)
    time.sleep(1)

    while puzzle.grid != puzzle.goal:
        valid_moves = puzzle.get_valid_moves()
        chosen_move = random.choice(valid_moves)
        puzzle.move(chosen_move)
        moves_count += 1

        
    print(f"Solved in {moves_count} moves!")

ReflexAgent()

Initial Grid:
[7, 2, 4]
[5, 0, 6]
[8, 3, 1]

Solved in 324034 moves!


In [13]:
Grid = [7, 2, 4, 5, 0, 6, 8, 3, 1]
Goal = [0, 1, 2, 3, 4, 5, 6, 7, 8]

def ModelAgent():
    """Model-based agent - avoids reversing last move."""
    puzzle = Puzzle(Grid, Goal)
    moves_count = 0
    last_move_undo = None
    opposites = {'up': 'down', 'down': 'up', 'left': 'right', 'right': 'left'}

    print("Initial Grid:")
    printG(puzzle.grid)
    time.sleep(1)

    while puzzle.grid != puzzle.goal:
        possible_moves = puzzle.get_valid_moves()
        
        if last_move_undo in possible_moves and len(possible_moves) > 1:
            possible_moves.remove(last_move_undo)

        chosen_move = random.choice(possible_moves)
        last_move_undo = opposites[chosen_move]
        puzzle.move(chosen_move)
        moves_count += 1
        
    print(f"Solved in {moves_count} moves!")

ModelAgent()

Initial Grid:
[7, 2, 4]
[5, 0, 6]
[8, 3, 1]

Solved in 105714 moves!


In [14]:
# ==================== GOAL-BASED AGENT (A* Search) ====================
Grid = [7, 2, 4, 5, 0, 6, 8, 3, 1]
Goal = [0, 1, 2, 3, 4, 5, 6, 7, 8]

def GoalBasedAgent():
    """Goal-based agent using A* search with Manhattan distance."""
    start = tuple(Grid)
    goal = tuple(Goal)
    
    # Priority queue: (f_score, g_score, state, path)
    frontier = [(calculate_manhattan_distance(Grid, Goal), 0, start, [])]
    explored = set()
    
    print("Initial Grid:")
    printG(Grid)
    print("Searching for solution...\n")
    time.sleep(1)
    
    while frontier:
        f, g, state, path = heapq.heappop(frontier)
        
        if state in explored:
            continue
        explored.add(state)
        
        if state == goal:
            print(f"Solved in {len(path)} moves!")
            print(f"Path: {' -> '.join(path)}")
            printG(list(state))
            return
        
        # Generate neighbors
        grid = list(state)
        puzzle = Puzzle(grid, Goal)
        
        for move in puzzle.get_valid_moves():
            temp = grid[:]
            i = temp.index(0)
            
            if move == "up": swap = i - 3
            elif move == "down": swap = i + 3
            elif move == "left": swap = i - 1
            else: swap = i + 1
            
            temp[i], temp[swap] = temp[swap], temp[i]
            next_state = tuple(temp)
            
            if next_state not in explored:
                new_g = g + 1
                h = calculate_manhattan_distance(temp, Goal)
                new_f = new_g + h
                heapq.heappush(frontier, (new_f, new_g, next_state, path + [move]))

GoalBasedAgent()

Initial Grid:
[7, 2, 4]
[5, 0, 6]
[8, 3, 1]

Searching for solution...

Solved in 26 moves!
Path: left -> up -> right -> down -> down -> left -> up -> right -> right -> up -> left -> left -> down -> right -> right -> down -> left -> left -> up -> right -> right -> up -> left -> down -> left -> up
[0, 1, 2]
[3, 4, 5]
[6, 7, 8]



In [23]:
# ==================== UTILITY-BASED AGENT (A* Search) ====================

Grid = [7, 2, 4, 5, 0, 6, 8, 3, 1]
Goal = [0, 1, 2, 3, 4, 5, 6, 7, 8]

def calculate_utility(grid, goal, moves):
    """Calculate utility: higher is better."""
    manhattan = calculate_manhattan_distance(grid, goal)
    misplaced = sum(1 for i in range(9) if grid[i] != 0 and grid[i] != goal[i])
    utility = 100.0 / (manhattan + 2) - moves * 0.8 - misplaced
    return utility

def UtilityBasedAgent():
    """Utility-based agent using A* search with utility function."""
    start = tuple(Grid)
    goal = tuple(Goal)
    
    # Priority queue: (-utility, g_score, state, path)
    start_utility = calculate_utility(Grid, Goal, 0)
    frontier = [(-start_utility, 0, start, [])]
    explored = set()
    
    print("Initial Grid:")
    printG(Grid)
    print("Searching for solution...\n")
    time.sleep(1)
    
    while frontier:
        neg_u, g, state, path = heapq.heappop(frontier)
        
        if state in explored:
            continue
        explored.add(state)
        
        if state == goal:
            print(f"Solved in {len(path)} moves!")
            print(f"Path: {' -> '.join(path)}")
            printG(list(state))
            return
        
        # Generate neighbors
        grid = list(state)
        puzzle = Puzzle(grid, Goal)
        
        for move in puzzle.get_valid_moves():
            temp = grid[:]
            i = temp.index(0)
            
            if move == "up": swap = i - 3
            elif move == "down": swap = i + 3
            elif move == "left": swap = i - 1
            else: swap = i + 1
            
            temp[i], temp[swap] = temp[swap], temp[i]
            next_state = tuple(temp)
            
            if next_state not in explored:
                new_g = g + 1
                utility = calculate_utility(temp, Goal, new_g)
                heapq.heappush(frontier, (-utility, new_g, next_state, path + [move]))

UtilityBasedAgent()

Initial Grid:
[7, 2, 4]
[5, 0, 6]
[8, 3, 1]

Searching for solution...

Solved in 26 moves!
Path: left -> up -> right -> down -> right -> down -> left -> left -> up -> right -> right -> down -> left -> left -> up -> right -> right -> up -> left -> left -> down -> right -> right -> up -> left -> left
[0, 1, 2]
[3, 4, 5]
[6, 7, 8]

